# try-except-solve — worked example 2: List-batched safe solve

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `try-except-solve`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When solving a Python list of independent systems, a per-item `try/except RuntimeError` lets one singular matrix fail without aborting the loop. Each output slot is either the solution or `None`, and output length always equals input length.

## Worked solution

We solve a list of `(A, b)` pairs, tolerating singular entries.

1. We loop over `zip(As, bs)` so each system is handled independently.
2. Each `t.linalg.solve` is guarded by `try/except RuntimeError`; a singular `A` appends `None` and the loop continues to the next pair.
3. Because every iteration appends exactly one result, the output list length matches the input length and order is preserved.

The printed results show solved vectors interleaved with `None` exactly where the inputs were singular.

In [ ]:
import torch as t
from typing import Optional, List

def safe_solve_list(As, bs) -> List[Optional[t.Tensor]]:
    out = []
    for A, b in zip(As, bs):
        try:
            out.append(t.linalg.solve(A, b))
        except RuntimeError:
            out.append(None)
    return out

As = [t.tensor([[3.0, 0.0], [0.0, 1.0]]), t.tensor([[1.0, 1.0], [1.0, 1.0]])]
bs = [t.tensor([9.0, 2.0]), t.tensor([1.0, 1.0])]
res = safe_solve_list(As, bs)
print('first:', res[0].tolist())
print('second (singular):', res[1])
print('length preserved:', len(res) == len(As))